# 03 - Modelado final y pipeline reproducible


## 1. Configuración del repositorio

El notebook detecta automáticamente la raíz de `proyecto-solaris` tanto si Jupyter se abre desde la raíz como desde `notebooks/`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "solaris_datos_limpios.csv"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Raíz del proyecto:", PROJECT_ROOT)
print("Dataset procesado:", DATA_PROCESSED)


## 2. Verificación de la entrada

`solaris_datos_limpios.csv` debe haber sido generado previamente por el notebook `02_model_selection.ipynb`.

Este notebook no vuelve a limpiar los datos crudos: utiliza directamente la versión procesada y versionada.


In [ ]:
if not DATA_PROCESSED.exists():
    raise FileNotFoundError(
        "No se encontró data/processed/solaris_datos_limpios.csv. "
        "Ejecuta primero 02_model_selection.ipynb."
    )

print("Archivo procesado encontrado correctamente.")


## 3. Importación del pipeline final

Las funciones se importan desde `src/pipeline.py`, que constituye la fuente única de verdad para el entrenamiento final.


In [ ]:
from src.pipeline import (
    RUTA_DATOS,
    RUTA_CONFUSION,
    RUTA_PIPELINE,
    RUTA_METRICAS,
    configurar_reproducibilidad,
    cargar_datos,
    definir_features_target,
    auditar_leakage,
    dividir_datos,
    construir_preprocesador,
    construir_pipeline,
    optimizar_pipeline,
    evaluar_modelo,
    verificar_leakage_post_entrenamiento,
    guardar_artefactos,
    imprimir_resumen,
)

print("Pipeline importado correctamente.")
print("Entrada configurada :", RUTA_DATOS)
print("Modelo final         :", RUTA_PIPELINE)
print("Métricas             :", RUTA_METRICAS)
print("Matriz de confusión  :", RUTA_CONFUSION)


## 4. Reproducibilidad y carga de datos


In [ ]:
configurar_reproducibilidad()

df = cargar_datos(RUTA_DATOS)

print("\nDimensiones del dataset procesado:", df.shape)
df.head()


## 5. Definición de features y target

El problema se mantiene como clasificación binaria:

- `0`: cliente **Al día**
- `1`: cliente **Moroso**

Los clientes con estado `Cancelado` no forman parte de las dos clases objetivo.


In [ ]:
(
    X,
    y,
    numeric_features,
    categorical_features,
) = definir_features_target(df)

print("\nX:", X.shape)
print("y:", y.shape)


## 6. Auditoría pre-split de data leakage

La revisión se realiza antes de dividir los datos para confirmar que las variables seleccionadas pueden utilizarse bajo los supuestos temporales documentados.


In [ ]:
auditoria = auditar_leakage()

auditoria


## 7. División Train/Test

Se mantiene un split estratificado 80/20 con `random_state=42`.


In [ ]:
(
    X_train,
    X_test,
    y_train,
    y_test,
) = dividir_datos(X, y)

print("\nTrain:", X_train.shape)
print("Test :", X_test.shape)


## 8. Construcción del preprocesamiento y del modelo

El preprocesamiento y Random Forest quedan encapsulados dentro de un único `sklearn Pipeline`.


In [ ]:
preprocessor = construir_preprocesador(
    numeric_features,
    categorical_features,
)

pipeline = construir_pipeline(preprocessor)

pipeline


## 9. Optimización con GridSearchCV

La búsqueda evalúa las combinaciones definidas en `src/pipeline.py` mediante validación cruzada de 5 folds y utiliza ROC-AUC como métrica de selección.


In [ ]:
search, best_pipeline = optimizar_pipeline(
    pipeline,
    X_train,
    y_train,
)

print("\nMejores hiperparámetros:")
print(search.best_params_)

print("\nMejor ROC-AUC CV:")
print(round(search.best_score_, 4))


## 10. Evaluación final en test

El conjunto test permanece aislado hasta esta etapa.


In [ ]:
resultados_test = evaluar_modelo(
    best_pipeline,
    X_test,
    y_test,
)

{
    "ROC-AUC Test": round(resultados_test["auc_test"], 4),
    "Precision Moroso": round(resultados_test["precision_test"], 4),
    "Recall Moroso": round(resultados_test["recall_test"], 4),
    "F1 Moroso": round(resultados_test["f1_test"], 4),
    "Accuracy": round(resultados_test["accuracy_test"], 4),
}


## 11. Verificación post-entrenamiento

Se compara ROC-AUC de train y test. La metodología del proyecto utiliza una diferencia absoluta superior a `0.10` como señal de alerta que requiere revisión.


In [ ]:
(
    auc_train,
    auc_test,
    diferencia_auc,
    estado_leakage,
) = verificar_leakage_post_entrenamiento(
    best_pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
)

print("ROC-AUC Train :", round(auc_train, 4))
print("ROC-AUC Test  :", round(auc_test, 4))
print("Diferencia    :", round(diferencia_auc, 4))
print("Estado        :", estado_leakage)


## 12. Serialización y artefactos

El pipeline completo, las métricas y la matriz de confusión se guardan automáticamente en las carpetas profesionales del repositorio.


In [ ]:
metricas = guardar_artefactos(
    best_pipeline,
    search,
    y_test,
    resultados_test,
    auc_train,
    auc_test,
    diferencia_auc,
    estado_leakage,
)

print("\nArtefactos generados:")
print("- Modelo :", RUTA_PIPELINE)
print("- Métricas:", RUTA_METRICAS)
print("- Figura :", RUTA_CONFUSION)


## 13. Resumen final


In [ ]:
imprimir_resumen(
    search,
    auc_train,
    auc_test,
    diferencia_auc,
    resultados_test,
    estado_leakage,
)


## Conclusión

Este notebook constituye la etapa final del flujo reproducible de SOLARIS. La preparación de datos se realiza previamente en el notebook 02 y el entrenamiento final reutiliza directamente el código versionado de `src/pipeline.py`.

De esta manera, la ejecución desde Jupyter y la ejecución por terminal:

```bash
python src/pipeline.py
```

utilizan la misma lógica de modelado y generan los mismos tipos de artefactos.
